# RAG Customer Support Assistant
### (LangGraph + Human-in-the-Loop)

This notebook implements a full Retrieval-Augmented Generation (RAG) system using:
- PDF Knowledge Base
- ChromaDB (Vector Database)
- OpenAI Embeddings + LLM
- LangGraph Workflow
- Human-in-the-Loop (HITL)


In [ ]:
# Install dependencies
!pip install langchain langchain-community langchain-openai chromadb pypdf langgraph -q

In [ ]:
# Import libraries
import os
from typing import TypedDict

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

from langgraph.graph import StateGraph

In [ ]:
# Set your OpenAI API Key
os.environ['OPENAI_API_KEY'] = 'sk-xxxx'

In [ ]:
# Upload PDF
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load PDF
loader = PyPDFLoader('support.pdf')
documents = loader.load()

print(documents[0].page_content[:300])

In [ ]:
# Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print('Total chunks:', len(chunks))

In [ ]:
# Embeddings + ChromaDB
embedding_model = OpenAIEmbeddings()

vector_db = Chroma.from_documents(chunks, embedding_model, persist_directory='./chroma_db')
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

In [ ]:
# LLM Setup
llm = ChatOpenAI()

def generate_answer(query, docs):
    context = '\n'.join([doc.page_content for doc in docs])

    prompt = f"""
    Answer ONLY using the context.

    Context:
    {context}

    Question: {query}
    """

    return llm.invoke(prompt).content

In [ ]:
# Graph State
class GraphState(TypedDict):
    query: str
    context: str
    answer: str
    confidence: float

In [ ]:
# Processing Node
def process_node(state):
    query = state['query']
    docs = retriever.get_relevant_documents(query)

    if not docs:
        return {'context': '', 'answer': '', 'confidence': 0.0}

    context = '\n'.join([d.page_content for d in docs])
    answer = generate_answer(query, docs)
    confidence = len(docs) / 3

    return {'context': context, 'answer': answer, 'confidence': confidence}

In [ ]:
# Human-in-the-Loop Node
def human_node(state):
    print('\nEscalated to Human Support')
    response = input('Enter human response: ')
    return {'answer': response}

In [ ]:
# Output Node
def output_node(state):
    return {'answer': state['answer']}

In [ ]:
# Routing Logic
def route(state):
    if state['confidence'] < 0.3:
        return 'human'
    return 'output'

In [ ]:
# Build LangGraph
graph = StateGraph(GraphState)

graph.add_node('process', process_node)
graph.add_node('human', human_node)
graph.add_node('output', output_node)

graph.set_entry_point('process')

graph.add_conditional_edges(
    'process',
    route,
    {'human': 'human', 'output': 'output'}
)

app = graph.compile()

In [ ]:
# Run the system
query = input('Ask your question: ')
result = app.invoke({'query': query})

print('\nFinal Answer:')
print(result['answer'])